# P124 — Redes de atención sobre grafos

## 1. Título y paper

**Paper:** *Graph Attention Networks*  
**Autoría:** Petar Veličković, Guillem Cucurull, Arantxa Casanova, Adriana Romero, Pietro Liò, Yoshua Bengio  
**Año y venue:** 2018 · ICLR 2018 · arXiv:1710.10903  
**Nivel:** L3 · **Motor:** `gat`  
**Ficha completa:** [`P124_gat`](../../papers/foundational/P124_gat/README.md)

**Hito:** Sustituye el promedio uniforme sobre los vecinos por pesos aprendidos por pareja, sin necesitar conocer la estructura global del grafo.

- [arXiv:1710.10903](https://arxiv.org/abs/1710.10903)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La convolución de grafo promedia a todos los vecinos por igual y normaliza por el grado. Eso supone que todos los vecinos importan lo mismo y exige conocer el grafo completo, lo que impide aplicar el modelo a nodos que no se vieron al entrenar.
2. Ejecutar una implementación mínima de la propuesta: Calcular un coeficiente de atención para cada pareja de nodos vecinos, normalizarlo con softmax sobre el vecindario y agregar con esos pesos. Varias cabezas en paralelo, como en el Transformer.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P08
- P120


## 4. Intuición

La convolución de grafo promedia a todos los vecinos por igual. Si la mitad son ruido, el promedio también lo es. La atención decide cuánto pesa cada vecino, por pareja.


## 5. Concepto mínimo

```text
GCN : h'ᵢ = σ( Σⱼ  (1/√(dᵢdⱼ)) · W·hⱼ )        peso FIJO por grado
GAT : h'ᵢ = σ( Σⱼ  αᵢⱼ · W·hⱼ )                 peso APRENDIDO por pareja

     αᵢⱼ = softmax_j( LeakyReLU(aᵀ[W·hᵢ ‖ W·hⱼ]) )
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('gat', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Gana la atención con un grafo limpio?
2. ¿Y con seis vecinos ruidosos por cada tres útiles?
3. ¿Cuánto peso le da al vecino más discrepante?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('gat', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('gat', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con el grafo limpio, la media uniforme es **ligeramente mejor** (1,0 frente a 0,963): ponderar solo puede desequilibrar un promedio que ya era correcto. Con **6 vecinos ruidosos**, la media cae a **0,744** y la atención se sostiene en **0,869**. Al vecino más discrepante le da **0,003** donde el peso uniforme sería 0,167.


## 10. Comentario pedagógico

Ese primer resultado es el más instructivo: **atender tiene un coste cuando no hay nada que filtrar**. La atención no es gratis ni universalmente mejor; gana exactamente donde el grafo es sucio, que es el caso real. Y como los pesos se calculan por pareja y no dependen de la estructura global, el modelo se aplica a nodos que no se vieron al entrenar.


## 11. Error o anti-patrón deliberado

Anti-patrón: interpretar los pesos de atención como una explicación de la decisión.


In [ ]:
print('Un peso alto dice que ese vecino influyo, no por que.')
print('Hay trabajos que muestran que las atenciones aprendidas acaban casi uniformes.')
print('Usalos como diagnostico, no como justificacion ante nadie.')

## 12. Corrección

El barrido de ruido y los pesos de un caso:


In [ ]:
r = run_paper_lab('gat', seed=3)['result']
for fila in r['por_nivel_de_ruido']:
    print(fila)
print('ejemplo:', r['ejemplo'])

## 13. Desafío guiado

Explica en qué condiciones la atención NO aporta nada sobre el promedio uniforme, y cómo lo comprobarías antes de complicar un modelo.


In [ ]:
r = run_paper_lab('gat', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma un grafo tuyo, mide la fracción de aristas que unen clases distintas y decide si la atención te compensaría. Justifícalo con el barrido del motor.


## 15. Evidencia de aprendizaje

Guarda la fracción medida y tu decisión razonada.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P124_gat/README.md) · evaluación formal: [`assessments/papers/P124_gat.md`](../../assessments/papers/P124_gat.md)


## 16. Cierre

Grafos resueltos. Queda el documento: texto donde la posición en la página es parte del significado.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
